# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook explores the FAIR^2 clinical dataset using the `mlcroissant` library, demonstrating how to load, overview, and analyze data described by a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list all RecordSets and display their fields and columns, referencing all items by their `@id`.

In [ ]:
# List all RecordSets, their @id, and the fields/columns they contain.
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No RecordSets found in the metadata. Please check schema structure.")
else:
    print("Available RecordSets:")
    for rs in record_sets:
        print(f"\nRecordSet name: {rs.name}\n@id: {rs.id}")

        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for fld in rs.fields:
                print(f"    - {fld.name} (@id: {fld.id}, type: {getattr(fld, 'data_type', None)})")
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - {col.name} (@id: {col.id}, type: {getattr(col, 'data_type', None)})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the RecordSet and Field `@id`s from the overview.

In [ ]:
# Build DataFrames for all available RecordSets by @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")

# Show column names for the first RecordSet and a preview
if record_set_ids:
    sample_id = record_set_ids[0]
    print(f"\nColumns in RecordSet @id {sample_id}:")
    print(dataframes[sample_id].columns.tolist())
    dataframes[sample_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. We'll select numeric and categorical fields by `@id` for demonstration based on the earlier overview.

In [ ]:
# Choose a RecordSet that holds patient/clinical data (update these IDs as needed from overview above)
main_rs_id = record_set_ids[0] if record_set_ids else None

if main_rs_id is not None:
    df = dataframes[main_rs_id]
    # Try to automatically detect a likely numeric field
    numeric_candidates = df.select_dtypes(include=['number']).columns
    if len(numeric_candidates) == 0:
        print("No numeric fields found for analysis.")
    else:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")

        # Filtering
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df = filtered_df.copy()  # avoid SettingWithCopyWarning
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to detect a groupable (categorical) field
        possible_cats = df.select_dtypes(include=['object', 'category']).columns
        group_field = None
        for col in possible_cats:
            num_unique = df[col].nunique()
            if 1 < num_unique < 10:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped by {group_field}, mean {numeric_field}:")
            display(grouped_df)
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No main RecordSet available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization for numeric_field (histogram and boxplot)
if main_rs_id is not None and len(numeric_candidates) > 0:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1,2,2)
    sns.boxplot(data=df, x=numeric_field)
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()

    # If group_field is available, make a boxplot per group
    if group_field is not None:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()


## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 clinical dataset described by a Croissant schema using the `mlcroissant` library. We identified all record sets and fields by their `@id`, created DataFrames, performed filtering and normalization on numeric data, and visualized field distributions.

Further analysis can be carried out by referencing additional `@id`s for columns or fields of interest as demonstrated. This approach ensures reproducibility and clarity when working with FAIR datasets in a machine-actionable way.